In [1]:
import json
import numpy as np
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer
from collections import Counter
import torch
from torch.utils.data import DataLoader, TensorDataset
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal

In [2]:
CLIENT = OpenAI(base_url="http://127.0.0.1:8001/v1", api_key="none")

In [3]:
def call_llm(system_prompt, user_query, response_model=None):
    api_params = {
        "model": "Qwen/Qwen2.5-14B-Instruct",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ],
        "max_tokens": 8192,
        "temperature": 0.2,
    }

    if response_model:
        api_params["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": response_model.__name__,
                "schema": response_model.model_json_schema(),
            },
        }

    response = CLIENT.chat.completions.create(**api_params)
    return response.choices[0].message.content

## Dataset creation

dataset format: 

[entity, sentence] -> label

It will predict the label given an entity and the sentence it occurs in. each entity can have different labels when existing in different sentences. from the text we will predict the labels of all entities and merge all the labels of a single entity in a list

In [4]:

LabelType = Literal[
    "Time",
    "Activity",
    "Location",
    "Food",
    "Object",
    "Text",
    "Medical_Concept",
    "Sanskrit_text",
    "Social_Group_&_Role",
    "Phenomenon",
    "Concept",
    "Event",
    "Primordial_Element",
    "Geographical_Feature",
    "Emotions",
    "Body_part",
    "Living_Being",
    "Celestial_Entity",
    "Mythical_Entity"
]

class EntityClassification(BaseModel):
    label: LabelType = Field(..., description="Exactly one label from the predefined set")

In [5]:
entity_defintions = '''
1.  Mythical_Entity - It includes entities like deities, gods, their reincarnations/avatars, demons, असुर, देवता, and entities which are personified in the text and are shown to be worshiped
2.  Celestial_Entity - It includes celestial bodies like sun, moon, stars, constellations and their derivative concepts like नक्षत्र and zodiacs.
3.  Phenomenon - It includes events that happen or happened either in nature or on celestial level. it includes seasons. It does not include man-made events.
4.  Time - Any word/event/entity that marks a specific point or period of time. It includes but is not limited to date, day, month, year, lunar phases, युग (like कलयुग), etc. It does not includes festivals.
5.  Food - Any object or thing that is or can be eaten or drunk. It includes names of dishes as well as items used to prepare them.
6.  Activity - It includes any kind of Ritual that is carried out, or festival that is celebrated, or any other kind of mundane actions or activities someone can indulge in.
        Do NOT classify major, unique historical or epic milestones here. (e.g., 'Sleeping/Eating/Praying' is an Activity; 'The Kurukshetra War' is an Event).

7.  Concept - Intangible, abstract ideas, philosophies, or states of being that exist in the mind or as universal principles, and cannot be physically touched or localized in physical space.

        Includes (Sub-types):
        a. Abstract Outcomes & Paradigms: Universal conditions or results of actions (e.g., Victory, Defeat, Freedom).
        b. Beliefs & Philosophies: Moral, religious, or philosophical frameworks and doctrines (e.g., Dharma, Adharma, Karma, Ahimsa).
        c. Knowledge & Disciplines: Fields of study, sciences, or bodies of wisdom (e.g., Ayurveda, Science, Mathematics, Malla-vidya).
        d. States & Conditions: The temporary or permanent intangible state of an entity or system (e.g., Sleep, Decay, Chaos, Peace, Poverty).
        e. Attributes & Qualities: Intangible characteristics, traits, or properties belonging to other entities (e.g., Beauty, Strength, Courage, Redness, Wisdom).

        Negative Constraints (What it is NOT):
        a. Do NOT classify tangible, physical objects here (e.g., a "Coin/सिक्का" is an Object; "Wealth/संपत्ति" is a Concept. An "Idol/मूर्ति" is an Object; "Devotion/भक्ति" is a Concept).
        b. Do NOT classify specific chronological events or festivals here (e.g., "Diwali" is a Festival/Event; "Celebration" is a Concept).
        c. Do NOT classify specific human emotions here. We have a separate "Emotion" category (e.g., "Anger" goes to Emotion, not Concept).
        
        Examples of Entities in this Category: * विजय (Victory), पराजय (Defeat), धर्म (Dharm), पाप (Sin), ज्ञान (Knowledge), शक्ति (Strength), निद्रा (Sleep), आयुर्वेद (Ayurveda).

8.  Object - it includes any non-living tangible entity including any artifact, idols, structure, figure, drawing, instrument, tool, weapon, material, derivative etc. of either natural or man-made origin. It can not include food and books/text/scriptures as they have separate categories.
9.  Living_Being - It includes single instance of any person, animal, or plant, either named or unnamed.
        Do NOT classify divine, supernatural, or worshiped figures here unless the text is talking about an earthly incarnation acting as a mortal, you are allowed to give them both tags. Use this primarily for standard mortals, ordinary animals, and flora. 
        (e.g., 'A horse/अश्व' is a Living_Being; 'Uchchaihshravas/उच्चैःश्रवा' [Indra's divine horse] is a Mythical_Entity)

10. Text - It includes any named or unnamed written work including religious works, epics and written treatise on any subject.
11. Location - It includes any named or unnamed man-made place and any directions and positions like east, west, left, right, etc. Do not include names of Celestial_Entity, Days of week, objects and Geographical_Features.
12. Primordial_Element - The fundamental, uncompounded physical substances, cosmic forces, and primal states of energy that serve as the foundational building blocks of the material universe. These are the base ingredients of nature, not the complex events created by them.

        Includes (Sub-types):
        a. The Classical Elements: The raw, base forms of matter and space (e.g., Earth/Soil, Water, Fire, Air, Sky/Aether/Space).
        b. Primal Energies: Fundamental states of cosmic illumination or thermodynamics (e.g., Light, Darkness, Heat, Cold, Shadow).

        Negative Constraints (What it is NOT):
        a. Do NOT classify derivative weather patterns, natural disasters, or atmospheric events here. (e.g., "Air/वायु" is a Primordial Element, but a "Storm/आंधी" is a Natural Phenomenon. "Fire/अग्नि" is an Element, but a "Forest Fire/दावानल" is an Event).
        b. Do NOT classify specific geographical bodies here. (e.g., "Water/जल" is an Element, but a "River/नदी" or "Ocean/समुद्र" is a Geographical Entity).
        c. Do NOT classify specific celestial bodies here. (e.g., "Light/प्रकाश" is an Element, but the "Sun/सूर्य" is a Celestial Object).
        d. Do NOT classify periodic table elements (like Gold, Iron, or Copper) here; those belong in Physical Objects or Materials.

        Examples of Entities in this Category: * अग्नि (Fire), जल (Water), वायु (Air), आकाश (Sky/Space), प्रकाश (Light), अन्धकार (Darkness), शीत (Cold), ऊष्मा (Heat), पृथ्वी (Earth/as a base substance).

13. Medical_Concept - Includes any entity that represents the bodily conditions and states of any Living_Being. It includes any medical conditions, diseases, their symptoms, states like hunger, thirst, blood-pressure, unconsciousness, etc. as well.
14. Geographical_Feature - Naturally occurring, large-scale physical landforms, bodies of water, and terrestrial ecosystems that shape the surface of the Earth. These are structural features of nature, distinct from the raw chemical elements that make them up.

        Includes (Sub-types):
        a. Topography & Landforms: Mountains, valleys, hills, caves, plateaus, islands (e.g., पर्वत, गुफा, घाटी).
        b. Bodies of Water: Rivers, oceans, lakes, waterfalls, ponds (e.g., नदी, समुद्र, सरोवर, झरना).
        c. Ecosystems & Biomes: Forests, jungles, deserts, grasslands, swamps (e.g., वन, अरण्य, मरुस्थल).

        Negative Constraints (What it is NOT):
        a. Do NOT classify Primordial Elements here (e.g., a "River/नदी" is a Geographical Feature, but the "Water/जल" inside it is a Primordial_Element).
        b. Do NOT classify specific, individual plants or animals here (e.g., a "Forest/वन" is a Geographical Feature, but a single "Banyan Tree/बरगद" is a Living_Being).
        c. Do NOT classify purely political, administrative, or man-made settlements here unless they are intrinsically tied to the natural landscape (e.g., a "City/नगर" or "Kingdom/राज्य" is strictly a Location or Social_Structure, whereas a "Mountain/पर्वत" is a Geographical Feature).

        Examples of Entities in this Category: * हिमालय (Himalayas), गंगा (Ganges), वन (Forest), पर्वत (Mountain), गुफा (Cave), समुद्र (Ocean).
    
15. Event - Major historical, narrative, or epic milestones that define the plot or history of the world. These are unique, named, or highly significant occurrences (e.g., सेतुबन्धन, लंका दहन, महाभारत युद्ध). Negative Constraint: Do NOT classify routine daily actions or standard rituals here; those belong to Activity.
16. Emotions - Conscious affective states, subjective psychological feelings, and inner moods experienced by a living being. It includes primal feelings such as Anger, Fear, Joy, Sorrow, and Disgust, nuanced psychological states like Compassion, Jealousy, Arrogance, Infatuation, and Greed and Temperamental States of the mind like Calmness, Restlessness, or Confusion.
        
        Negative Constraints (What it is NOT):
        a. Do NOT classify the physical expression or action of an emotion here. (e.g., "Crying/रुदन" or "Laughing/अट्टहास" is an Activity; "Sorrow/शोक" or "Joy/हर्ष" is an Emotion).
        b. Do NOT classify physiological sensations or physical suffering here. (e.g., "Hunger/भूख", "Thirst/प्यास", or "Physical Pain/पीड़ा" belong to Medical_Concept; "Mental Anguish/संताप" or "Fear/भय" is an Emotion).
        c. Do NOT classify broad philosophical, religious, or universal paradigms here. (e.g., "Devotion/भक्ति" is a Concept; "Love/प्रेम" is an Emotion. "Sin/पाप" is a Concept; "Guilt/ग्लानि" is an Emotion).

        Examples of Entities in this Category: * क्रोध (Anger), प्रेम (Love), शोक (Sorrow/Grief), भय (Fear), मोह (Attachment/Infatuation), ईर्ष्या (Jealousy), करुणा (Compassion), शांति (Mental Calmness), अहंकार (Pride/Ego).

17. Social_Group_&_Role - The categorization of people based on their societal position, occupation, lineage, or religious affiliation. It represents a collective community identity, a professional occupation, or a hierarchical title held by an individual.

        Includes (Sub-types):
        a. Religious Sects & Lineages (सम्प्रदाय और गोत्र): Groups defined by their faith or ancestral line (e.g., Shaiv, Vaishnav, Suryavanshi, Raghukul).
        b. Social Hierarchy & Caste (वर्ण और जाति): The traditional societal classifications (e.g., Brahmin, Kshatriya, Vaishya, Shudra, Chandal).
        c. Occupations & Professions (व्यवसाय): Work-based identities or roles (e.g., Priest/पुरोहित, Charioteer/सूत, Warrior/योद्धा, Weaver/जुलाहा).
        d. Societal Titles & Positions (पद और उपाधि): Hierarchical or relational roles within society or a kingdom (e.g., King/राजा, Minister/मंत्री, Guru/गुरु, Disciple/शिष्य, Queen/रानी).

        Negative Constraints (What it is NOT):
        a. Do NOT classify specific, named individuals here. (e.g., "King/राजा" is a Social Role; "Dasharatha/दशरथ" is a Living_Being).
        b. Do NOT classify the abstract philosophy or practice here. (e.g., "A Shaiva/शैव" [the follower] is a Social Role; "Shaivism/शैव दर्शन" is a Concept).
        c. Do NOT classify mythological species/races here if they belong in the Mythical category. (e.g., "Asura/असुर" and "Devata/देवता" strictly belong to Mythical_Entity as per earlier rules, not here).
        d. Do NOT classify the physical governing body or geographic kingdom here. (e.g., "The Kingdom/राज्य" is a Location; "The King/राजा" is a Social Role).

        Examples of Entities in this Category: * शैव (Shaiv), ब्राह्मण (Brahmin), रघुकुल (Raghukul), राजा (Raja), पुरोहित (Purohit), क्षत्रिय (Kshatriya), शिष्य (Disciple), सारथी (Charioteer), मंत्री (Minister)

18. Body_part - It includes any organ or part of any Living_Being.
19. Sanskrit_text - It includes any kind of sanskrit mantra or shlokas mentioned in the text.

'''


In [6]:
def safe_llm_classify(entity, sentence, options):
    options_str = json.dumps(options, ensure_ascii=False, indent=2)

    task = (
        "Select the SINGLE most appropriate category for the entity "
        "based strictly on the given sentence. Do not infer beyond the sentence."
    )

    user_query = f"""
Entity: {entity}

Sentence: {sentence}

Options:
{options_str}

Return exactly one label.
"""

    try:
        response = call_llm(
            system_prompt=f"{task}\n\n{entity_defintions}",
            user_query=user_query,
            response_model=EntityClassification
        )

        parsed = EntityClassification.model_validate_json(response)
        return parsed.label

    except Exception:
        # fallback: choose first known label (safe default)
        return options[0]

In [7]:
with open("processing_state.json", "r", encoding="utf-8") as f:
    file = json.load(f)

In [8]:
entt_to_cat = dict()
for k, v_list in file['global_entity_registry'].items():
    for v in v_list:
        if v not in entt_to_cat:
            entt_to_cat[v] = []
        entt_to_cat[v].append(k)

In [10]:
dataset = []

direct_count = 0
llm_call_count = 0

for i in range(5):
    with open(f"entity_output_files/chapter_{i}_output.json", "r", encoding="utf-8") as f:
        entity_file = json.load(f)

    for group in entity_file:
        for obj in group['extracted_entities']:
            sentence = obj['sentence']
            entities = obj['entities']

            for entt in entities:
                labels = entt_to_cat.get(entt, [])

                # skip if no label known
                if not labels:
                    continue

                # case 1: single label → use ground truth directly
                if len(labels) == 1:
                    direct_count += 1
                    dataset.append({
                        "entity": entt,
                        "sentence": sentence,
                        "label": labels[0]
                    })
                    continue

                # case 2: multiple possible labels → disambiguate using LLM
                # IMPORTANT: restrict to known labels only
                selected_label = safe_llm_classify(entt, sentence, labels)
                llm_call_count += 1

                # extra safety: ensure label is valid
                if selected_label not in labels:
                    selected_label = labels[0]

                dataset.append({
                    "entity": entt,
                    "sentence": sentence,
                    "label": selected_label
                })

In [11]:
print(direct_count, llm_call_count)

2504 1103


In [12]:
dataset[1]

{'entity': 'सूर्योदय',
 'sentence': 'संवत्सरारम्भ के लिए सूर्योदय व्यापिनी प्रतिपदा लेनी चाहिए।',
 'label': 'Phenomenon'}

In [40]:
with open("classifier_data/dataset.json", 'w', encoding='utf-8') as f:
    json.dump(dataset, f, indent=4, ensure_ascii=False)

## Training Simple Classifier

In [13]:
def format_input(entity, sentence):
    return f"Entity: {entity} [SEP] Sentence: {sentence}"

In [14]:
label_list = [
    'Time','Activity','Location','Food','Object','Text',
    'Medical_Concept','Sanskrit_text','Social_Group_&_Role',
    'Phenomenon','Concept','Event','Primordial_Element',
    'Geographical_Feature','Emotions','Body_part','Living_Being',
    'Celestial_Entity','Mythical_Entity'
]

label2id = {l:i for i,l in enumerate(label_list)}
id2label = {i:l for l,i in label2id.items()}

In [15]:
X = [format_input(d["entity"], d["sentence"]) for d in dataset]
y = [label2id[d["label"]] for d in dataset]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

In [16]:
counts = Counter(y_train)
total = len(y_train)

class_weights = torch.tensor([
    total / counts[i] for i in range(len(label_list))
], dtype=torch.float)

In [17]:
# %%
encoder = SentenceTransformer("krutrim-ai-labs/Vyakyarth")

X_train_emb = encoder.encode(X_train, convert_to_numpy=True, show_progress_bar=True)
X_val_emb   = encoder.encode(X_val, convert_to_numpy=True, show_progress_bar=True)
X_test_emb  = encoder.encode(X_test, convert_to_numpy=True, show_progress_bar=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/91 [00:00<?, ?it/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

In [18]:
# %%
X_train_t = torch.tensor(X_train_emb, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

X_val_t = torch.tensor(X_val_emb, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.long)

X_test_t = torch.tensor(X_test_emb, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

In [19]:
# %%
train_ds = TensorDataset(X_train_t, y_train_t)
val_ds   = TensorDataset(X_val_t, y_val_t)
test_ds  = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=32)
test_loader  = DataLoader(test_ds, batch_size=32)

In [20]:

import torch.nn as nn

class Classifier(nn.Module):
    def __init__(self, input_dim, num_labels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_labels)
        )

    def forward(self, x):
        return self.net(x)

In [21]:
# %%
input_dim = X_train_t.shape[1]
num_labels = len(label_list)

model = Classifier(input_dim, num_labels)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [22]:
# %%
def evaluate(model, loader):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item()

    return total_loss / len(loader)


best_val = float("inf")
patience = 5
counter = 0

epochs = 60

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)
    val_loss = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}: Train={train_loss:.4f}, Val={val_loss:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        best_state = model.state_dict()
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping")
        break

model.load_state_dict(best_state)

Epoch 1: Train=2.8912, Val=2.8250
Epoch 2: Train=2.7605, Val=2.7020
Epoch 3: Train=2.6259, Val=2.5584
Epoch 4: Train=2.4834, Val=2.4324
Epoch 5: Train=2.3388, Val=2.3175
Epoch 6: Train=2.2261, Val=2.2244
Epoch 7: Train=2.1439, Val=2.1500
Epoch 8: Train=2.0605, Val=2.0830
Epoch 9: Train=1.9918, Val=2.0419
Epoch 10: Train=1.9461, Val=1.9881
Epoch 11: Train=1.9039, Val=1.9514
Epoch 12: Train=1.8650, Val=1.9211
Epoch 13: Train=1.8259, Val=1.8886
Epoch 14: Train=1.7989, Val=1.8691
Epoch 15: Train=1.7716, Val=1.8517
Epoch 16: Train=1.7223, Val=1.8343
Epoch 17: Train=1.7351, Val=1.8186
Epoch 18: Train=1.6853, Val=1.8043
Epoch 19: Train=1.6840, Val=1.7998
Epoch 20: Train=1.6568, Val=1.7832
Epoch 21: Train=1.6456, Val=1.7695
Epoch 22: Train=1.6058, Val=1.7664
Epoch 23: Train=1.6136, Val=1.7660
Epoch 24: Train=1.5934, Val=1.7586
Epoch 25: Train=1.5568, Val=1.7565
Epoch 26: Train=1.5597, Val=1.7569
Epoch 27: Train=1.5628, Val=1.7541
Epoch 28: Train=1.5647, Val=1.7481
Epoch 29: Train=1.4975, Val=1

<All keys matched successfully>

In [23]:
# %%
from sklearn.metrics import classification_report

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for xb, yb in test_loader:
        logits = model(xb)
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.numpy())
        all_labels.extend(yb.numpy())

print(classification_report(all_labels, all_preds, target_names=label_list))

                      precision    recall  f1-score   support

                Time       0.57      0.41      0.48        49
            Activity       0.48      0.23      0.31        57
            Location       0.11      0.17      0.13         6
                Food       0.43      0.57      0.49        21
              Object       0.26      0.36      0.30        14
                Text       0.07      0.33      0.12         6
     Medical_Concept       0.30      0.67      0.41         9
       Sanskrit_text       0.20      0.33      0.25         3
 Social_Group_&_Role       0.12      0.25      0.17         8
          Phenomenon       0.25      0.56      0.34         9
             Concept       0.53      0.10      0.17        96
               Event       0.20      1.00      0.33         2
  Primordial_Element       0.43      0.50      0.46         6
Geographical_Feature       1.00      1.00      1.00         2
            Emotions       0.12      0.40      0.18         5
       

## FineTuning MuRIL

In [24]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")

In [25]:
class MurilDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=128,
            return_tensors='pt'
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [26]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "google/muril-base-cased",
    num_labels=len(label_list)
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

In [33]:
from transformers import Trainer
import torch.nn as nn

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [28]:
for param in model.base_model.parameters():
    param.requires_grad = False

### stage 1

In [37]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./muril_stage1",
    learning_rate=5e-4,   # higher since encoder frozen
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True
)

In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=MurilDataset(X_train, y_train),
    eval_dataset=MurilDataset(X_val, y_val)
)

trainer.train()

OutOfMemoryError: CUDA out of memory. Tried to allocate 16.00 MiB. GPU 4 has a total capacity of 47.40 GiB of which 12.81 MiB is free. Process 3234981 has 2.87 GiB memory in use. Process 1779076 has 43.33 GiB memory in use. Including non-PyTorch memory, this process has 1.17 GiB memory in use. Of the allocated memory 776.63 MiB is allocated by PyTorch, and 7.37 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)